In [0]:
import sys
sys.path.append("/Workspace/Users/bm5726@rit.edu/Drafts/snowflake-healthcare-normalization/src")

from data_processor import DataProcessor

processor = DataProcessor("/Volumes/workspace/healthcare/raw/legacy_healthcare_data.csv")
processor.process_all()

In [0]:
%sql
SHOW TABLES IN workspace.healthcare;

In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM workspace.healthcare.factvisit) AS fact_rows,
  (SELECT COUNT(*) FROM workspace.healthcare.factvisit f
     LEFT JOIN workspace.healthcare.dimpatient p ON f.patient_id = p.patient_id
     WHERE p.patient_id IS NULL) AS orphan_patients,
  (SELECT COUNT(*) FROM workspace.healthcare.factvisit f
     LEFT JOIN workspace.healthcare.dimprovider d ON f.provider_id = d.provider_id
     WHERE d.provider_id IS NULL) AS orphan_providers,
  (SELECT COUNT(*) FROM workspace.healthcare.factvisit f
     LEFT JOIN workspace.healthcare.dimbilling b ON f.billing_id = b.billing_id
     WHERE b.billing_id IS NULL) AS orphan_billing;

In [0]:
%sql
WITH checks AS (
  SELECT
    (SELECT COUNT(*) FROM workspace.healthcare.factvisit) AS fact_rows,
    SUM(CASE WHEN f.patient_id IS NOT NULL AND p.patient_id IS NULL THEN 1 ELSE 0 END) AS orphan_patient,
    SUM(CASE WHEN f.insurance_id IS NOT NULL AND i.insurance_id IS NULL THEN 1 ELSE 0 END) AS orphan_insurance,
    SUM(CASE WHEN f.billing_id IS NOT NULL AND b.billing_id IS NULL THEN 1 ELSE 0 END) AS orphan_billing,
    SUM(CASE WHEN f.provider_id IS NOT NULL AND pr.provider_id IS NULL THEN 1 ELSE 0 END) AS orphan_provider,
    SUM(CASE WHEN f.location_id IS NOT NULL AND l.location_id IS NULL THEN 1 ELSE 0 END) AS orphan_location,
    SUM(CASE WHEN f.primary_diagnosis_id IS NOT NULL AND pd.primary_diagnosis_id IS NULL THEN 1 ELSE 0 END) AS orphan_primary_dx,
    SUM(CASE WHEN f.secondary_diagnosis_id IS NOT NULL AND sd.secondary_diagnosis_id IS NULL THEN 1 ELSE 0 END) AS orphan_secondary_dx,
    SUM(CASE WHEN f.treatment_id IS NOT NULL AND t.treatment_id IS NULL THEN 1 ELSE 0 END) AS orphan_treatment,
    SUM(CASE WHEN f.prescription_id IS NOT NULL AND rx.prescription_id IS NULL THEN 1 ELSE 0 END) AS orphan_prescription,
    SUM(CASE WHEN f.lab_order_id IS NOT NULL AND lo.lab_order_id IS NULL THEN 1 ELSE 0 END) AS orphan_lab_order
  FROM workspace.healthcare.factvisit f
  LEFT JOIN workspace.healthcare.dimpatient p              ON f.patient_id = p.patient_id
  LEFT JOIN (SELECT DISTINCT insurance_id FROM workspace.healthcare.diminsurance) i ON f.insurance_id = i.insurance_id
  LEFT JOIN workspace.healthcare.dimbilling b              ON f.billing_id = b.billing_id
  LEFT JOIN workspace.healthcare.dimprovider pr            ON f.provider_id = pr.provider_id
  LEFT JOIN workspace.healthcare.dimlocation l             ON f.location_id = l.location_id
  LEFT JOIN workspace.healthcare.dimprimarydiagnosis pd    ON f.primary_diagnosis_id = pd.primary_diagnosis_id
  LEFT JOIN workspace.healthcare.dimsecondarydiagnosis sd  ON f.secondary_diagnosis_id = sd.secondary_diagnosis_id
  LEFT JOIN workspace.healthcare.dimtreatment t            ON f.treatment_id = t.treatment_id
  LEFT JOIN workspace.healthcare.dimprescription rx        ON f.prescription_id = rx.prescription_id
  LEFT JOIN workspace.healthcare.dimlaborder lo            ON f.lab_order_id = lo.lab_order_id
)
SELECT stack(11,
  'fact_rows', fact_rows,
  'orphan_patient', orphan_patient,
  'orphan_insurance', orphan_insurance,
  'orphan_billing', orphan_billing,
  'orphan_provider', orphan_provider,
  'orphan_location', orphan_location,
  'orphan_primary_dx', orphan_primary_dx,
  'orphan_secondary_dx', orphan_secondary_dx,
  'orphan_treatment', orphan_treatment,
  'orphan_prescription', orphan_prescription,
  'orphan_lab_order', orphan_lab_order
) AS (check_name, result)
FROM checks;

In [0]:
%sql
WITH raw AS (
  SELECT visit_id, secondary_diagnosis_code AS raw_code
  FROM read_files('/Volumes/workspace/healthcare/raw/legacy_healthcare_data.csv',
                  format => 'csv', header => true)
)
SELECT
  COUNT(*) AS visits_checked,
  SUM(CASE WHEN r.raw_code IS NOT NULL AND r.raw_code = d.secondary_diagnosis_code THEN 1 ELSE 0 END) AS correct_match,
  SUM(CASE WHEN r.raw_code IS NOT NULL AND (d.secondary_diagnosis_code IS NULL OR r.raw_code <> d.secondary_diagnosis_code) THEN 1 ELSE 0 END) AS wrong_match,
  SUM(CASE WHEN r.raw_code IS NULL AND f.secondary_diagnosis_id IS NULL THEN 1 ELSE 0 END) AS correctly_empty
FROM workspace.healthcare.factvisit f
JOIN raw r ON f.visit_id = r.visit_id
LEFT JOIN workspace.healthcare.dimsecondarydiagnosis d
  ON f.secondary_diagnosis_id = d.secondary_diagnosis_id;

In [0]:
display(spark.table("workspace.healthcare.factvisit"))